# VideoFake — codice aggiornato da GitHub

[Repository](https://github.com/LucaLorenz/VideoFake) · Basato su [VideoFake di NeuroModern](https://github.com/neuromodern/VideoFake).

Seleziona un runtime **GPU T4**, poi esegui gli step in ordine. Lo step 0 definisce il collegamento a GitHub: gli step successivi scaricano il codice dell'ultimo commit ogni volta che premi Esegui.

Dopo una correzione nella repo, riesegui solo lo step indicato. Le celle restano semplici comandi di avvio; il codice aggiornato vive in `colab_runner.py`. Se cambiano le dipendenze, il messaggio chiederà di rieseguire lo step 3.

In caso di errore fermati: condividi il solo blocco `DEBUG_BEGIN … DEBUG_END`, oppure usa **File → Salva una copia in GitHub** includendo gli output. I log completi restano nel runtime. Nessun invio automatico degli errori.

L'ambiente usa Python 3.10 separato; non reinstallare CUDA/cuDNN di sistema. La prima installazione può richiedere diversi minuti.


In [ ]:
#@title 0. Collega Colab al codice aggiornato su GitHub
import json
import time
import types
import urllib.request

def _vf_step(step, **config):
    global VF_DEBUG
    revision = 'non disponibile'
    try:
        api = 'https://api.github.com/repos/LucaLorenz/VideoFake/git/ref/heads/main'
        request = urllib.request.Request(api + '?t=' + str(time.time_ns()),
            headers={'User-Agent': 'VideoFake-Colab', 'Cache-Control': 'no-cache'})
        with urllib.request.urlopen(request, timeout=30) as response:
            revision = json.load(response)['object']['sha']
        url = f'https://raw.githubusercontent.com/LucaLorenz/VideoFake/{revision}/colab_runner.py'
        with urllib.request.urlopen(url, timeout=30) as response:
            code = response.read().decode('utf-8')
        runner = types.ModuleType('videofake_live_runner')
        exec(compile(code, 'colab_runner.py@' + revision[:12], 'exec'), runner.__dict__)
        VF_DEBUG = runner.run_step(step, revision, config)
    except Exception as error:
        VF_DEBUG = {'revision': revision, 'step': step, 'status': 'error',
                    'error': f'{type(error).__name__}: {error}'}
        print('DEBUG_BEGIN\n' + json.dumps(VF_DEBUG, ensure_ascii=False, indent=2) + '\nDEBUG_END')
    return VF_DEBUG

VF_DEBUG = _vf_step('diagnose')


In [ ]:
#@title 1. Scarica applicazione
VF_DEBUG = _vf_step("repo")


In [ ]:
#@title 2. Prepara cartelle
VF_DEBUG = _vf_step("folders")


In [ ]:
#@title 3. Installa dipendenze e verifica GPU
VF_DEBUG = _vf_step("setup")


In [ ]:
#@title 4. Imposta file di ingresso e uscita
source = "/content/videofake-live/app/face.jpg" #@param {type:"string"}
target = "/content/videofake-live/app/input_video.mp4" #@param {type:"string"}
output = "/content/videofake-output/out.mp4" #@param {type:"string"}
print('Percorsi impostati. Carica i tuoi file oppure usa gli esempi del progetto.')


In [ ]:
#@title 5. Esegui usando l'ultimo codice su GitHub
Device = "cuda" #@param ["cuda", "cpu"]
Processor = "face_swapper face_enhancer" #@param ["face_swapper face_enhancer", "face_swapper", "face_enhancer"]
VideoEncoder = "libx264" #@param ["libx264", "libx265", "libvpx-vp9"]
VideoQuality = "18" #@param {type:"string"}
KeepFPS = True #@param {type:"boolean"}
KeepAudio = True #@param {type:"boolean"}
KeepFrames = False #@param {type:"boolean"}
ManyFaces = True #@param {type:"boolean"}
VF_DEBUG = _vf_step('run', source=source, target=target, output=output,
    Device=Device, Processor=Processor, VideoEncoder=VideoEncoder, VideoQuality=VideoQuality,
    KeepFPS=KeepFPS, KeepAudio=KeepAudio, KeepFrames=KeepFrames, ManyFaces=ManyFaces)


In [ ]:
#@title 6. Mostra ultimo rapporto breve
print('DEBUG_BEGIN\n' + json.dumps(VF_DEBUG, ensure_ascii=False, indent=2) + '\nDEBUG_END')
